# Week 3 · Diagnose and Improve Training

**Activities 2–4** Complete paper activity 1 first; return to worksheet activity 5 afterwards. The worksheet and this notebook contain different tasks.

Work in pairs to allow for some discussion. Predict → change a setting → inspect → explain. Derivatives are supplied explicitly. Run cells in order using the repository's Python environment. All cells run as supplied: an unfinished warm-up prints a reminder without blocking later activities. Later activities do not depend on solving the warm-up.

Data, models, and plots are supplied. Change the clearly marked controls, not the data-generation seed or split during a comparison. Write answers in the **Your record** cells. For environment setup help, see `README.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
    "text.color": "#202020", "axes.labelcolor": "#202020",
    "xtick.color": "#202020", "ytick.color": "#202020", "font.size": 10})
DTYPE = torch.float64
print("Ready: CPU calculations; NumPy", np.__version__, "PyTorch", torch.__version__)


## 2. Repair the optimizer

### 2A. Complete one update
We reuse $L(u,v)=\tfrac12(u^2+10v^2)$. Replace `None` in `student_step` by a gradient-descent expression using `theta`, `grad`, and `lr`. If you leave it unfinished, the cell prints a reminder and you can continue with the supplied experiments. The check uses a point different from the paper example.

This small function is for practice; the subsequent experiment uses its own supplied implementation.

In [ ]:
def student_step(theta, grad, lr):
    candidate = None  # TODO: replace None by the parameter-update expression.
    return candidate

probe = np.array([2., -1.])
probe_gradient = np.array([2., -10.])
candidate = student_step(probe, probe_gradient, .05)
if candidate is None:
    print("Warm-up not completed yet. Replace None when ready; you may continue to the experiments.")
else:
    np.testing.assert_allclose(candidate, [1.9, -.5])
    print("Your update passed the check.")


In [ ]:
def quadratic_run(lr, beta=0., steps=40):
    theta = np.array([1., 1.])
    velocity = np.zeros(2)
    path = [theta.copy()]
    for _ in range(steps):
        grad = theta * np.array([1., 10.])
        velocity = beta * velocity + grad
        theta = theta - lr * velocity
        path.append(theta.copy())
        if not np.isfinite(theta).all() or np.linalg.norm(theta) > 1e6:
            break  # Keep an unstable experiment from overflowing.
    path = np.array(path)
    losses = .5 * (path[:, 0]**2 + 10 * path[:, 1]**2)
    return path, losses

def plot_quadratic(runs):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    u, v = np.meshgrid(np.linspace(-1.5, 1.5, 160), np.linspace(-1.5, 1.5, 160))
    axes[0].contour(u, v, .5 * (u*u + 10*v*v), levels=[.05,.2,.5,1,2,4,8,12], colors="0.75")
    for name, (path, losses) in runs.items():
        axes[0].plot(path[:,0], path[:,1], '.-', markersize=3, label=name)
        axes[1].semilogy(np.arange(len(losses)), np.maximum(losses, 1e-16), label=name)
    axes[0].plot(0, 0, 'k*', markersize=10)
    axes[0].set(xlabel="u", ylabel="v", xlim=(-1.5,1.5), ylim=(-1.5,1.5), title="Parameter paths (fixed viewing window)")
    axes[1].set(xlabel="Updates", ylabel="Loss", title="Loss includes the initial value")
    axes[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()


### 2B. Test your learning-rate predictions
Before running, copy your one-step results for 0.01, 0.1, and 0.25 from the worksheet and predict what further updates will do. Inspect both the contour plot and loss curves. A divergent path can leave the fixed contour window; it has not disappeared from training.

1. Which predictions were supported? Identify slow progress, a coordinate reaching zero immediately, and divergence.
2. Add learning rate 0.15 and inspect the path. Does crossing from one side of zero to the other necessarily imply increasing loss?
3. Compare all runs from the same starting point and with the same update budget. Explain why these controls matter.

In [ ]:
# CHANGE THESE CONTROLS, then rerun this cell.
learning_rates = [.01, .1, .25]
steps = 40
plain_runs = {f"GD: lr={lr}": quadratic_run(lr, steps=steps) for lr in learning_rates}
plot_quadratic(plain_runs)


**Your record — 2B:** predictions / observations / explanation of alternating coordinates versus loss.

_Write here._


### 2C. Add gradient history
Write the parameter vector as $\theta=(u,v)$. Use $m_t$ for the momentum accumulator, so it is distinct from the coordinate $v$. The supplied rule is $m_{t+1}=\beta m_t+\nabla L(\theta_t)$, $\theta_{t+1}=\theta_t-\eta m_{t+1}$, with $m_0=0$. The landscape remains fixed; momentum changes the path across it.

Predict and compare these settings: `(lr=0.15, beta=0)`, `(0.15, 0.4)`, and `(0.4, 0.9)`. The last changes **two** settings and is a stress test, not an isolated test of momentum. To isolate momentum, compare the first two.

1. Compare the final loss and the first update reaching loss below 0.01. If it is never reached, record that result.
2. Explain one beneficial effect and one failure using gradient history. Does reaching a threshold guarantee remaining below it?
3. If time permits, change only beta to 0.9 while holding lr=0.15. Is more momentum always better?

We compare updates for this tiny example, not wall-clock performance or universal optimizer rankings.

In [ ]:
# CHANGE THESE CONTROLS. Keep the same initial point and number of updates.
settings = [(0.15, 0.0), (0.15, 0.4), (0.4, 0.9)]
momentum_runs = {f"lr={lr}, beta={beta}": quadratic_run(lr, beta, steps=40)
                 for lr, beta in settings}
plot_quadratic(momentum_runs)
for name, (path, losses) in momentum_runs.items():
    hit = np.flatnonzero(losses < .01)
    print(name, "final loss:", f"{losses[-1]:.3e}",
          "first update below .01:", int(hit[0]) if len(hit) else "not reached")


**Your record — 2C:** settings / first update to threshold / final loss / explanation / limitation of the comparison.

_Write here._


## 3. How noisy should an update be?

We now fit a linear model to **synthetic data**: three informative features and 37 irrelevant features, plus measurement noise. With only 50 training observations, fitting noise is possible even with a linear model. These are not polynomial features and no external dataset is downloaded.

The fitted model is

$$\hat y=b+w_1x_1+\cdots+w_{40}x_{40}.$$

There are **41 fitted parameters**: 40 weights and an intercept. Before standardization, the synthetic targets are generated as $y=1.5x_1-x_2+0.5x_3+\epsilon$, where $\epsilon$ is random noise. The other 37 features have no effect on the generated target, but a model can still fit accidental patterns involving them in a small training sample.

The fixed split has 50 training, 90 validation, and 100 reserved test observations. Feature scaling uses training statistics only. Test data are reserved and never evaluated in this session. The intercept is the first parameter. All reported losses are half MSE; L2, when enabled later, is $\lambda\|w\|^2$ on weights only.

The complete training and plotting code below is supplied. Read the comments identifying the gradient, sampling, and update; you need not implement it. Gradients are calculated explicitly and assigned to PyTorch's SGD optimizer, so using it requires no backpropagation knowledge.

In [ ]:
def make_data(seed=31):
    rng = np.random.default_rng(seed)
    raw = rng.normal(size=(240, 40))
    truth = np.zeros(40); truth[:3] = [1.5, -1., .5]
    target = raw @ truth + rng.normal(size=240)
    mu, scale = raw[:50].mean(axis=0), raw[:50].std(axis=0)
    features = (raw - mu) / scale
    design = np.column_stack([np.ones(240), features])
    tensor = lambda a: torch.tensor(a, dtype=DTYPE)
    return (tensor(design[:50]), tensor(target[:50]),
            tensor(design[50:140]), tensor(target[50:140]))

X_train, y_train, X_valid, y_valid = make_data()
print("Training:", tuple(X_train.shape), "Validation:", tuple(X_valid.shape),
      "100 test observations reserved, not exposed to the training experiment.")

def fit(lr=.05, epochs=20, batch_size=50, seed=0, lam=0.):
    n = len(y_train)
    theta = torch.zeros(X_train.shape[1], dtype=DTYPE, requires_grad=True)
    optimizer = torch.optim.SGD([theta], lr=lr)
    rng = np.random.default_rng(seed)
    train_loss, valid_loss, weights = [], [], []
    def record():
        with torch.no_grad():
            train_loss.append(float(.5 * ((X_train @ theta - y_train)**2).mean()))
            valid_loss.append(float(.5 * ((X_valid @ theta - y_valid)**2).mean()))
            weights.append(theta.detach().numpy().copy())
    record()
    updates = 0
    for _ in range(epochs):
        order = rng.permutation(n)  # A fresh shuffle each epoch; no replacement within it.
        for start in range(0, n, batch_size):
            ids = order[start:start + batch_size]  # Retain the final partial batch.
            xb, yb = X_train[ids], y_train[ids]
            with torch.no_grad():
                penalty_grad = 2 * lam * theta.detach().clone()
                penalty_grad[0] = 0.  # Intercept is unpenalized.
                grad = xb.T @ (xb @ theta - yb) / len(ids) + penalty_grad
            optimizer.zero_grad()
            theta.grad = grad
            optimizer.step()  # theta <- theta - lr * grad
            updates += 1
            record()  # Evaluate FULL train/validation losses after every update.
    return {"train": np.array(train_loss), "valid": np.array(valid_loss),
            "weights": np.array(weights), "updates": updates,
            "updates_per_epoch": int(np.ceil(n / batch_size))}


Predict which batch size will introduce the most variability. Run the comparison, then repeat it with a different shuffle seed.

1. Compare full training-loss curves; these are not losses on whichever batch was just sampled.
2. Record the updates per epoch and total updates for each batch size. What is controlled by equal epochs, and what is not?
3. Does the comparison isolate gradient noise as the only cause of different progress? Explain why not.
4. State a conclusion supported across your runs, avoiding “this batch size is always best.”

In [ ]:
# CHANGE THESE CONTROLS: first compare batches, then change only shuffle_seed.
batch_sizes = [1, 10, 50]
shuffle_seed = 0
batch_epochs = 12
batch_runs = {b: fit(lr=.02, epochs=batch_epochs, batch_size=b, seed=shuffle_seed)
              for b in batch_sizes}
fig, ax = plt.subplots(figsize=(8, 4))
for b, run in batch_runs.items():
    epoch_axis = np.arange(len(run["train"])) / run["updates_per_epoch"]
    ax.plot(epoch_axis, run["train"], label=f"batch {b}")
    print(f"Batch {b}: {run['updates_per_epoch']} updates/epoch, {run['updates']} updates total")
ax.set(xlabel="Epochs (fractional positions mark updates)", ylabel="Full training half MSE")
ax.legend(); fig.tight_layout(); plt.show()


**Your record — 3:** prediction / results for at least two seeds / update counts / fair-comparison limitation.

_Write here._

## 4. A better fit—or a better model?

Use the same dataset, now with **full batches and fixed learning rate 0.05**. First inspect the supplied short-versus-long unregularized comparison. Then compare three L2 strengths. This changes the fitting objective, not merely the path to the same optimum.

1. Predict whether a lower training loss must imply a lower validation loss. Inspect updates 20 and 600.
2. Compare the final models at the same 600-update budget, including weight norms. Why do we report unpenalized half MSE rather than the different penalized objectives?
3. Now use the supplied validation-selected checkpoints. Among the three strengths and their checkpoints, record a preferred pair `(lambda, update)`. This is checkpoint selection from completed runs, **not an online patience-based stopping rule**.
4. Explain why this selection is not an unbiased final performance estimate. What would the reserved test data be used for after choices are frozen?

No guarantee is made that positive L2 beats every early unregularized checkpoint on this particular split.

In [ ]:
# Three guided settings. Additional tuning is optional, after the core discussion.
strengths = [0., .03, .3]
regularized_runs = {lam: fit(lr=.05, epochs=600, batch_size=50, lam=lam)
                    for lam in strengths}
base = regularized_runs[0.]
for t in [20, 600]:
    print(f"Unregularized update {t}: train={base['train'][t]:.4f}, validation={base['valid'][t]:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lam, run in regularized_runs.items():
    best = int(np.argmin(run["valid"]))
    axes[0].plot(run["train"], label=f"lambda={lam}")
    axes[1].plot(run["valid"], label=f"lambda={lam}")
    axes[1].scatter(best, run["valid"][best], s=25)
    norm = np.linalg.norm(run["weights"][-1, 1:])
    print(f"lambda={lam}: final train={run['train'][-1]:.4f}, final val={run['valid'][-1]:.4f}, "
          f"final weight norm={norm:.3f}, best checkpoint={best}, best val={run['valid'][best]:.4f}")
for ax, title in zip(axes, ["Training", "Validation (dots mark selected checkpoints)"]):
    ax.set(title=title, xlabel="Updates", ylabel="Half MSE"); ax.legend()
fig.tight_layout(); plt.show()


**Your record — 4:** short/long comparison / final-model comparison / selected strength and checkpoint / distinction between selection and final evaluation.

_Write here._

Return to **worksheet activity 5** for the final exercise.

### Sources
Original classroom tasks aligned with HealthML deep-learning-book Chapter 3. Data and landscapes are synthetic. Half MSE and L2 conventions match that chapter. Repository license: code MIT; prose and exercises CC BY-SA 4.0.